# Chapter 4 — Implementing Text Classification Using Logistic Regression

This notebook follows Chapter 4 of *Deep Learning for Natural Language Processing: A Gentle Introduction* by Mihai Surdeanu and Marco A. Valenzuela-Escárcega.

The chapter implements the text-classification concepts introduced in Chapters 2 and 3, beginning with binary classification and progressing to multiclass logistic regression using PyTorch.

## 4.1 Binary Classification

### 4.1.1 Large Movie Review Dataset

#### Large Movie Review Dataset

The binary classification task in this chapter uses the Large Movie Review Dataset from IMDb.

The dataset contains movie reviews with scores from 1 to 10. Reviews with scores above 6 are treated as positive, while reviews with scores below 5 are treated as negative. Reviews with scores of 5 or 6 are excluded because they are considered too neutral.

The dataset contains:

- 25,000 training reviews
- 25,000 test reviews
- Separate `pos` and `neg` directories for positive and negative reviews

Each review is stored in an individual text file.

In [1]:
from pathlib import Path
import tarfile
import urllib.request

# Local project paths
DATA_DIR = Path("data")
DATASET_DIR = DATA_DIR / "aclImdb"
ARCHIVE_PATH = DATA_DIR / "aclImdb_v1.tar.gz"

# Official Large Movie Review Dataset URL
DATASET_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

DATA_DIR.mkdir(parents=True, exist_ok=True)

if DATASET_DIR.exists():
    print(f"Dataset already exists at: {DATASET_DIR.resolve()}")
else:
    if not ARCHIVE_PATH.exists():
        print("Downloading IMDb dataset...")
        urllib.request.urlretrieve(DATASET_URL, ARCHIVE_PATH)
        print("Download complete.")

    print("Extracting dataset...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        tar.extractall(DATA_DIR)

    print(f"Dataset extracted to: {DATASET_DIR.resolve()}")

Download complete.
Extracting dataset...
Dataset extracted to: C:\Users\dubey\Documents\UC_GitLab\Deep_Learning_NLP_Surdeanu\Chapter_04_Text_Classification\data\aclImdb


#### Verify the Dataset Structure

Before loading the reviews, verify that the expected training and test directories exist and inspect the number of positive and negative examples in each split.

In [2]:
# Verify dataset directories and class counts

train_pos_dir = DATASET_DIR / "train" / "pos"
train_neg_dir = DATASET_DIR / "train" / "neg"
test_pos_dir = DATASET_DIR / "test" / "pos"
test_neg_dir = DATASET_DIR / "test" / "neg"

directories = {
    "Train Positive": train_pos_dir,
    "Train Negative": train_neg_dir,
    "Test Positive": test_pos_dir,
    "Test Negative": test_neg_dir,
}

for name, directory in directories.items():
    review_count = len(list(directory.glob("*.txt")))
    print(f"{name:15}: {review_count:,} reviews")

Train Positive : 12,500 reviews
Train Negative : 12,500 reviews
Test Positive  : 12,500 reviews
Test Negative  : 12,500 reviews


#### Inspect Sample Reviews and Filenames

Each review is stored as an individual text file. The filename contains both a review ID and the original IMDb rating.

For example, a filename such as `24_8.txt` indicates a review with ID `24` and an IMDb score of `8`.

We inspect one positive and one negative review to understand the raw data format before preprocessing.

In [3]:
# Inspect one positive and one negative training review

positive_file = next(train_pos_dir.glob("*.txt"))
negative_file = next(train_neg_dir.glob("*.txt"))

positive_text = positive_file.read_text(encoding="utf-8")
negative_text = negative_file.read_text(encoding="utf-8")

print("POSITIVE REVIEW")
print("Filename:", positive_file.name)
print("Text:")
print(positive_text[:1000])

print("\n" + "=" * 80 + "\n")

print("NEGATIVE REVIEW")
print("Filename:", negative_file.name)
print("Text:")
print(negative_text[:1000])

POSITIVE REVIEW
Filename: 0_9.txt
Text:
Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!


NEGATIVE REVIEW
Filename: 0_3.txt
Text:
Story of a man who has unnatural feelings for a pig. Starts out with a opening scene that is a terrific example

### 4.1.2 Bag-of-Words Model

Machine learning algorithms require numerical input, so the raw review text must be converted into feature vectors.

The Bag-of-Words (BoW) representation creates a vocabulary of words and represents each review using the frequency of those words. Word order is ignored; only the presence or frequency of vocabulary terms is retained.

#### From Text to a Document-Term Matrix

A Bag-of-Words representation first creates a vocabulary that assigns each unique word to a feature position. Each review is then converted into a vector containing the number of times each vocabulary word occurs.

For the full IMDb dataset, we use scikit-learn's `CountVectorizer` to automate vocabulary creation and text-to-vector conversion.

The resulting matrix is called a **document-term matrix**:

- each row represents one review (document)
- each column represents one vocabulary word (term)
- each cell contains the count of that word in that review

In [4]:
from glob import glob

pos_files = glob("data/aclImdb/train/pos/*.txt")
neg_files = glob("data/aclImdb/train/neg/*.txt")

print("number of positive reviews:", len(pos_files))
print("number of negative reviews:", len(neg_files))

number of positive reviews: 12500
number of negative reviews: 12500


#### Build the Training Document-Term Matrix

`CountVectorizer` converts the raw movie reviews into Bag-of-Words feature vectors.

For the training data, `fit_transform()` performs two operations:

1. `fit()` learns the vocabulary from the training reviews.
2. `transform()` converts each review into a vector of word counts.

The resulting **document-term matrix** has one row per review and one column per vocabulary term.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

# Learn the vocabulary from the training reviews
# and convert all reviews into Bag-of-Words vectors.
cv = CountVectorizer(input="filename")

doc_term_matrix = cv.fit_transform(pos_files + neg_files)

print("Matrix shape:", doc_term_matrix.shape)
print("Vocabulary size:", len(cv.vocabulary_))
print("Non-zero elements:", doc_term_matrix.nnz)
print("Matrix type:", type(doc_term_matrix))

Matrix shape: (25000, 74849)
Vocabulary size: 74849
Non-zero elements: 3445861
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>


#### Sparse Representation for Local Execution

The book converts the complete document-term matrix into a dense NumPy array using `toarray()` to simplify the downstream implementation.

For this dataset, however, a dense `int64` matrix would require approximately 14 GiB of memory. To avoid unnecessary memory usage in the local VS Code environment, we retain the document-term matrix in SciPy's sparse CSR format.

When an individual training example is required by the learning algorithm, that row can be converted to a dense NumPy vector. This changes only the storage strategy, not the Bag-of-Words representation or learning algorithm.

In [6]:
import numpy as np

# Estimate how much memory the dense representation would require
dense_elements = doc_term_matrix.shape[0] * doc_term_matrix.shape[1]
dense_bytes = dense_elements * doc_term_matrix.dtype.itemsize
dense_gib = dense_bytes / (1024 ** 3)

print(f"Dense elements: {dense_elements:,}")
print(f"Estimated dense memory: {dense_gib:.2f} GiB")

# Keep the training feature matrix sparse
X_train = doc_term_matrix

print("X_train shape:", X_train.shape)
print("X_train type:", type(X_train))

Dense elements: 1,871,225,000
Estimated dense memory: 13.94 GiB
X_train shape: (25000, 74849)
X_train type: <class 'scipy.sparse._csr.csr_matrix'>


#### Create the Training Labels

The document-term matrix was created by passing the positive review files first and the negative review files second.

Therefore, the corresponding target vector must contain:

- `1` for each positive review
- `0` for each negative review

The label at position `i` in `y_train` corresponds to the review represented by row `i` in `X_train`.

In [7]:
# Create training labels

y_pos = np.ones(len(pos_files))
y_neg = np.zeros(len(neg_files))

y_train = np.concatenate([y_pos, y_neg])

print("y_train shape:", y_train.shape)
print("First 5 labels:", y_train[:5])
print("Last 5 labels:", y_train[-5:])
print("Positive labels:", int(y_train.sum()))
print("Negative labels:", len(y_train) - int(y_train.sum()))

y_train shape: (25000,)
First 5 labels: [1. 1. 1. 1. 1.]
Last 5 labels: [0. 0. 0. 0. 0.]
Positive labels: 12500
Negative labels: 12500


### 4.1.3 Perceptron

We now implement the perceptron classifier introduced in Chapter 2.

The model contains:

- a weight vector `w`, with one weight for each Bag-of-Words feature
- a scalar bias `b`

Both parameters are initialized to zero and will be updated whenever the model misclassifies a training example.

In [8]:
# Initialize perceptron parameters

n_examples, n_features = X_train.shape

w = np.zeros(n_features)
b = 0

print("Number of training examples:", n_examples)
print("Number of features:", n_features)
print("Weight vector shape:", w.shape)
print("Initial bias:", b)

Number of training examples: 25000
Number of features: 74849
Weight vector shape: (74849,)
Initial bias: 0


#### Prepare the Perceptron Training Loop

The perceptron may require multiple passes through the training data. One complete pass through the training set is called an **epoch**.

The implementation uses a maximum of 10 epochs. At the beginning of each epoch, the training-example indices are shuffled so that the perceptron does not always see the reviews in the same order.

Only the indices are shuffled, which preserves the correspondence between each feature vector in `X_train` and its correct label in `y_train`.

In [9]:
# Configure perceptron training

n_epochs = 10
indices = np.arange(n_examples)

print("Maximum epochs:", n_epochs)
print("Number of training indices:", len(indices))
print("First 10 indices before shuffling:", indices[:10])

np.random.shuffle(indices)

print("First 10 indices after shuffling: ", indices[:10])

Maximum epochs: 10
Number of training indices: 25000
First 10 indices before shuffling: [0 1 2 3 4 5 6 7 8 9]
First 10 indices after shuffling:  [15733 14370   150 14901 15515   446  6522  2707  7590 21021]


#### Train the Perceptron

For each epoch, the training examples are shuffled and processed one at a time.

For each review:

1. Compute the perceptron score \(x \cdot w + b\).
2. Predict positive (`1`) if the score is greater than zero; otherwise predict negative (`0`).
3. If the prediction is correct, make no update.
4. If a positive review is incorrectly predicted as negative, add the review vector to the weights and increase the bias.
5. If a negative review is incorrectly predicted as positive, subtract the review vector from the weights and decrease the bias.

Training stops after the maximum number of epochs or earlier if an entire epoch contains no classification errors.

Because `X_train` is stored as a sparse matrix, only the non-zero feature weights are updated for each review.

In [10]:
from tqdm.auto import tqdm

# Train the perceptron

for epoch in range(n_epochs):
    n_errors = 0

    # Shuffle training-example order at the start of each epoch
    np.random.shuffle(indices)

    for i in tqdm(indices, desc=f"Epoch {epoch + 1}"):

        # Keep the current review as a sparse row
        x = X_train.getrow(i)
        y_true = int(y_train[i])

        # Perceptron decision
        score = float(x.dot(w)[0] + b)
        y_pred = 1 if score > 0 else 0

        # No update if prediction is correct
        if y_true == y_pred:
            continue

        # Positive review incorrectly predicted as negative
        if y_true == 1 and y_pred == 0:
            w[x.indices] += x.data
            b += 1

        # Negative review incorrectly predicted as positive
        elif y_true == 0 and y_pred == 1:
            w[x.indices] -= x.data
            b -= 1

        n_errors += 1

    print(f"Epoch {epoch + 1}: {n_errors:,} errors")

    # Convergence condition
    if n_errors == 0:
        print("Perceptron converged.")
        break

Epoch 1:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 1: 6,038 errors


Epoch 2:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 2: 4,331 errors


Epoch 3:   0%|          | 0/25000 [00:00<?, ?it/s]

...

Epoch 8: 2,523 errors


Epoch 9:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 9: 2,471 errors


Epoch 10:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 10: 2,286 errors


#### Prepare the Held-Out Test Dataset

After training, the perceptron is evaluated on the 25,000 held-out IMDb test reviews.

The test reviews must be represented using the same vocabulary learned from the training data. Therefore, we use `CountVectorizer.transform()` rather than `fit_transform()`.

Using `transform()` prevents the test data from changing the vocabulary and ensures that the training and test feature matrices have exactly the same columns.

In [11]:
# Load and transform the held-out test reviews

test_pos_files = glob("data/aclImdb/test/pos/*.txt")
test_neg_files = glob("data/aclImdb/test/neg/*.txt")

# IMPORTANT: use the vocabulary already learned from training data
X_test = cv.transform(test_pos_files + test_neg_files)

# Create test labels: positive = 1, negative = 0
y_test = np.concatenate([
    np.ones(len(test_pos_files)),
    np.zeros(len(test_neg_files))
])

print("Test positive reviews:", len(test_pos_files))
print("Test negative reviews:", len(test_neg_files))
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("X_test type:", type(X_test))

Test positive reviews: 12500
Test negative reviews: 12500
X_test shape: (25000, 74849)
y_test shape: (25000,)
X_test type: <class 'scipy.sparse._csr.csr_matrix'>


#### Predict Sentiment on the Test Set

The trained perceptron is now applied to the held-out test reviews.

For every review, the perceptron computes:

\[
score = x \cdot w + b
\]

A score greater than zero predicts a positive review (`1`); otherwise the review is predicted as negative (`0`).

Because `X_test` contains all test reviews, NumPy/SciPy can compute the scores for all 25,000 examples in a single matrix-vector operation.

In [12]:
# Make predictions on all test reviews

test_scores = X_test @ w + b
y_pred = test_scores > 0

print("Number of predictions:", len(y_pred))
print("First 10 scores:", test_scores[:10])
print("First 10 predictions:", y_pred[:10])
print("Predicted positives:", np.sum(y_pred))
print("Predicted negatives:", len(y_pred) - np.sum(y_pred))

Number of predictions: 25000
First 10 scores: [ 564. 1898. 1008.  755. 1431. 1234.  700.  344.  974. 2479.]
First 10 predictions: [ True  True  True  True  True  True  True  True  True  True]
Predicted positives: 14282
Predicted negatives: 10718


#### Evaluate the Perceptron

The predicted labels are compared with the true test labels using the confusion-matrix quantities:

- **True Positive (TP):** positive review correctly predicted as positive
- **False Positive (FP):** negative review incorrectly predicted as positive
- **True Negative (TN):** negative review correctly predicted as negative
- **False Negative (FN):** positive review incorrectly predicted as negative

These values are then used to calculate accuracy, precision, recall, and F1 score.

In [13]:
# Calculate confusion-matrix values manually

y_pred_int = y_pred.astype(int)

tp = np.sum((y_test == 1) & (y_pred_int == 1))
fp = np.sum((y_test == 0) & (y_pred_int == 1))
tn = np.sum((y_test == 0) & (y_pred_int == 0))
fn = np.sum((y_test == 1) & (y_pred_int == 0))

print("True Positives :", tp)
print("False Positives:", fp)
print("True Negatives :", tn)
print("False Negatives:", fn)
print("Total          :", tp + fp + tn + fn)

True Positives : 11553
False Positives: 2729
True Negatives : 9771
False Negatives: 947
Total          : 25000


#### Calculate Perceptron Evaluation Metrics

Using the confusion-matrix counts, we calculate:

- **Accuracy:** fraction of all predictions that are correct
- **Precision:** among predicted positive reviews, how many are actually positive
- **Recall:** among actual positive reviews, how many are correctly identified
- **F1 score:** harmonic mean of precision and recall

In [14]:
# Calculate evaluation metrics manually

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.8530
Precision: 0.8089
Recall   : 0.9242
F1 Score : 0.8627


#### Verify Metrics with scikit-learn

The manually calculated evaluation metrics are verified using scikit-learn's standard metric functions.

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print(f"Accuracy : {accuracy_score(y_test, y_pred_int):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_int):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_int):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_int):.4f}")

Accuracy : 0.8530
Precision: 0.8089
Recall   : 0.9242
F1 Score : 0.8627


### 4.1.4 Binary Logistic Regression from Scratch

We now implement binary logistic regression from scratch using NumPy.

Unlike the perceptron implementation, the book combines the feature weights and bias into a single weight vector. This is done by adding one extra feature to every training example whose value is always `1`.

The augmented representation is:

\[
x' = [x, 1]
\]

and the corresponding weight vector is:

\[
w' = [w, b]
\]

Therefore:

\[
w' \cdot x' = w \cdot x + b
\]

This allows the weights and bias to be updated together using the same gradient calculation.

In [16]:
from scipy.sparse import hstack, csr_matrix

# Add an always-on feature with value 1 to represent the bias
bias_feature = csr_matrix(np.ones((X_train.shape[0], 1)))

X_train_lr = hstack(
    [X_train, bias_feature],
    format="csr"
)

print("Original X_train shape:", X_train.shape)
print("Logistic regression shape:", X_train_lr.shape)
print("Matrix type:", type(X_train_lr))

Original X_train shape: (25000, 74849)
Logistic regression shape: (25000, 74850)
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>


#### Initialize Logistic Regression Parameters and Sigmoid

The augmented training matrix contains one feature for every vocabulary term plus one additional always-on bias feature.

A corresponding weight is initialized for each feature.

Logistic regression converts the raw linear score into a value between 0 and 1 using the sigmoid function:

\[
\sigma(z)=\frac{1}{1+e^{-z}}
\]

where:

\[
z = w \cdot x
\]

The sigmoid output is interpreted as the model's probability for the positive class.

In [18]:
# Initialize logistic regression weights

n_features_lr = X_train_lr.shape[1]
w_lr = np.random.random(n_features_lr)

print("Number of logistic regression features:", n_features_lr)
print("Weight vector shape:", w_lr.shape)
print("First 5 initial weights:", w_lr[:5])

Number of logistic regression features: 74850
Weight vector shape: (74850,)
First 5 initial weights: [0.34163981 0.87580777 0.67190493 0.52070437 0.10946349]


#### Implement the Logistic Function

Logistic regression converts the raw weighted score into a value between 0 and 1 using the logistic (sigmoid) function:

\[
\sigma(x)=\frac{1}{1+e^{-x}}
\]

A direct NumPy implementation can overflow when `exp(-x)` becomes too large. Therefore, we check the maximum value that can safely be represented by a `float64` and return `0.0` directly when the input would cause an overflow.

In [19]:
# Maximum representable float64 value
max_float = np.finfo(np.float64).max

def logistic(x):
    if -x > np.log(max_float):
        return 0.0

    return 1 / (1 + np.exp(-x))


# Test the logistic function
for value in [-5, 0, 5]:
    print(f"logistic({value:>2}) = {logistic(value):.6f}")

logistic(-5) = 0.006693
logistic( 0) = 0.500000
logistic( 5) = 0.993307


#### Train Binary Logistic Regression with SGD

The logistic regression model is trained using stochastic gradient descent (SGD).

For each training example:

1. Compute the linear score \(x \cdot w\).
2. Apply the logistic function to obtain the predicted probability.
3. Calculate the gradient:

\[
(\sigma - y)x
\]

4. Update the weights using:

\[
w \leftarrow w - \alpha(\sigma-y)x
\]

where \(\alpha\) is the learning rate.

The training examples are shuffled at the beginning of every epoch.

In [20]:
# Train binary logistic regression from scratch

learning_rate = 1e-1
n_epochs_lr = 10

indices_lr = np.arange(X_train_lr.shape[0])

for epoch in range(n_epochs_lr):

    # Randomize training-example order
    np.random.shuffle(indices_lr)

    for i in tqdm(indices_lr, desc=f"Epoch {epoch + 1}"):

        # Current training example
        x = X_train_lr.getrow(i)
        y = y_train[i]

        # 1. Linear decision score
        decision = float(x.dot(w_lr)[0])

        # 2. Predicted probability
        probability = logistic(decision)

        # 3. Gradient multiplier
        error = probability - y

        # 4. Update only the non-zero feature weights
        w_lr[x.indices] -= learning_rate * error * x.data

Epoch 1:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 6:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 7:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 8:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 9:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 10:   0%|          | 0/25000 [00:00<?, ?it/s]

#### Prepare the Test Matrix for Logistic Regression

The logistic regression model was trained using an augmented feature matrix containing one additional always-on feature for the bias.

The same feature must be added to the held-out test matrix so that each test example has exactly the same 74,850 features used during training.

In [21]:
# Add the bias feature to the test matrix

test_bias_feature = csr_matrix(
    np.ones((X_test.shape[0], 1))
)

X_test_lr = hstack(
    [X_test, test_bias_feature],
    format="csr"
)

print("Original X_test shape:", X_test.shape)
print("Logistic regression test shape:", X_test_lr.shape)
print("Weight vector shape:", w_lr.shape)

Original X_test shape: (25000, 74849)
Logistic regression test shape: (25000, 74850)
Weight vector shape: (74850,)


#### Predict with Binary Logistic Regression

For each test review, logistic regression first computes the linear score:

\[
z = x \cdot w
\]

The logistic function converts this score into a probability for the positive class:

\[
P(y=1 \mid x)=\sigma(z)
\]

A probability of at least `0.5` is classified as positive (`1`); otherwise it is classified as negative (`0`).

In [22]:
# Compute logistic regression probabilities on the test set

test_decisions_lr = X_test_lr @ w_lr

# Apply sigmoid element-wise
y_prob_lr = np.array([
    logistic(float(score))
    for score in test_decisions_lr
])

# Convert probabilities into binary predictions
y_pred_lr = (y_prob_lr >= 0.5).astype(int)

print("Number of probabilities:", len(y_prob_lr))
print("First 10 probabilities:", y_prob_lr[:10])
print("First 10 predictions:", y_pred_lr[:10])
print("Predicted positives:", np.sum(y_pred_lr))
print("Predicted negatives:", len(y_pred_lr) - np.sum(y_pred_lr))

Number of probabilities: 25000
First 10 probabilities: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
First 10 predictions: [1 1 1 1 1 1 1 1 1 1]
Predicted positives: 14363
Predicted negatives: 10637


#### Evaluate Binary Logistic Regression

The predicted labels are compared with the held-out test labels using accuracy, precision, recall, and F1 score.

These results allow us to compare the logistic regression classifier directly with the perceptron implemented in the previous subsection.

In [23]:
# Evaluate binary logistic regression

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

print(f"Accuracy : {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall   : {lr_recall:.4f}")
print(f"F1 Score : {lr_f1:.4f}")

Accuracy : 0.8508
Precision: 0.8053
Recall   : 0.9253
F1 Score : 0.8611


### 4.1.5 Binary Logistic Regression Utilizing PyTorch

We now implement the same binary logistic regression classifier using PyTorch.

PyTorch separates the implementation into three main components:

- `nn.Linear` represents the logistic regression model and contains the trainable weights and bias.
- `BCEWithLogitsLoss` computes the binary cross-entropy loss.
- `optim.SGD` applies stochastic gradient descent to update the model parameters.

Unlike the from-scratch implementation, we do not manually add a bias feature because `nn.Linear` already includes a trainable bias parameter.

In [24]:
import torch
from torch import nn
from torch import optim

# Hyperparameters
lr = 1e-1
n_epochs_torch = 10

# Logistic regression model
model = nn.Linear(n_features, 1)

# Binary cross-entropy loss
loss_func = nn.BCEWithLogitsLoss()

# Stochastic gradient descent optimizer
optimizer = optim.SGD(model.parameters(), lr=lr)

print(model)
print("Weight shape:", model.weight.shape)
print("Bias shape:", model.bias.shape)
print("Learning rate:", lr)

Linear(in_features=74849, out_features=1, bias=True)
Weight shape: torch.Size([1, 74849])
Bias shape: torch.Size([1])
Learning rate: 0.1


#### Inspect the Trainable PyTorch Parameters

The `nn.Linear` layer automatically creates the trainable parameters required by logistic regression:

- `weight`: one weight for each input feature
- `bias`: one scalar bias

These parameters are registered inside the model. Passing `model.parameters()` to the optimizer tells PyTorch which values should be updated during training.

In [25]:
# Inspect the trainable parameters

for name, parameter in model.named_parameters():
    print(f"Parameter: {name}")
    print(f"Shape    : {parameter.shape}")
    print(f"Requires gradient: {parameter.requires_grad}")
    print()

Parameter: weight
Shape    : torch.Size([1, 74849])
Requires gradient: True

Parameter: bias
Shape    : torch.Size([1])
Requires gradient: True



#### PyTorch Training Cycle

For each training example, PyTorch follows five main steps:

1. Clear previously stored gradients with `optimizer.zero_grad()`.
2. Compute the model output using `model(x)`.
3. Calculate the loss.
4. Compute gradients automatically using `loss.backward()`.
5. Update the model parameters using `optimizer.step()`.

This replaces the manual gradient calculation and weight update used in the from-scratch logistic regression implementation.

In [26]:
# Train binary logistic regression using PyTorch

indices_torch = np.arange(n_examples)

for epoch in range(n_epochs_torch):

    # Shuffle training examples
    np.random.shuffle(indices_torch)

    for i in tqdm(indices_torch, desc=f"Epoch {epoch + 1}"):

        # Convert only the current sparse review to a dense NumPy vector
        x_np = X_train.getrow(i).toarray().ravel().astype(np.float32)

        # Convert input and label to PyTorch tensors
        x = torch.from_numpy(x_np)
        y_true = torch.tensor(float(y_train[i]), dtype=torch.float32)

        # 1. Clear gradients from the previous example
        model.zero_grad()

        # 2. Forward pass: raw prediction score (logit)
        y_pred = model(x)

        # 3. Calculate binary cross-entropy loss
        loss = loss_func(y_pred[0], y_true)

        # 4. Automatically calculate gradients
        loss.backward()

        # 5. Update weights and bias using SGD
        optimizer.step()

Epoch 1:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 6:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 7:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 8:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 9:   0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 10:   0%|          | 0/25000 [00:00<?, ?it/s]

#### Evaluate the PyTorch Logistic Regression Model

After training, the PyTorch logistic regression model is evaluated on the held-out IMDb test set.

The model produces raw logits. The sigmoid function converts these logits into probabilities for the positive class, and a threshold of `0.5` converts the probabilities into binary predictions.

Evaluation uses accuracy, precision, recall, and F1 score.

In [27]:
# Evaluate the PyTorch logistic regression model

model.eval()

y_pred_torch = []

with torch.no_grad():
    for i in tqdm(range(X_test.shape[0]), desc="Evaluating"):

        # Convert one sparse test review to a dense float32 tensor
        x_np = X_test.getrow(i).toarray().ravel().astype(np.float32)
        x = torch.from_numpy(x_np)

        # Raw model score
        logit = model(x)

        # Convert logit to probability
        probability = torch.sigmoid(logit)

        # Apply 0.5 threshold
        prediction = int(probability.item() >= 0.5)

        y_pred_torch.append(prediction)

y_pred_torch = np.array(y_pred_torch)

torch_accuracy = accuracy_score(y_test, y_pred_torch)
torch_precision = precision_score(y_test, y_pred_torch)
torch_recall = recall_score(y_test, y_pred_torch)
torch_f1 = f1_score(y_test, y_pred_torch)

print(f"Accuracy : {torch_accuracy:.4f}")
print(f"Precision: {torch_precision:.4f}")
print(f"Recall   : {torch_recall:.4f}")
print(f"F1 Score : {torch_f1:.4f}")

Evaluating:   0%|          | 0/25000 [00:00<?, ?it/s]

Accuracy : 0.8549
Precision: 0.8236
Recall   : 0.9034
F1 Score : 0.8616


#### Binary Classification Results

Three binary sentiment classifiers were implemented and evaluated on the held-out IMDb test set.

| Model | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Perceptron | 0.8530 | 0.8089 | 0.9242 | 0.8627 |
| Logistic Regression — NumPy | 0.8508 | 0.8053 | 0.9253 | 0.8611 |
| Logistic Regression — PyTorch | 0.8549 | 0.8236 | 0.9034 | 0.8616 |

All three models achieved similar overall performance.

The PyTorch implementation demonstrates the main deep-learning training workflow:

\[
\text{forward pass}
\rightarrow
\text{loss}
\rightarrow
\text{backpropagation}
\rightarrow
\text{optimizer update}
\]

PyTorch automatically computes the gradients that were derived mathematically in Chapter 3 and implemented manually in the NumPy logistic regression model.

## 4.2 Multiclass Classification

### 4.2.1 AG News Dataset

The AG News dataset is used for multiclass text classification.

It contains four balanced news categories:

1. World
2. Sports
3. Business
4. Sci/Tech

The training set contains 120,000 articles and the test set contains 7,600 articles.

Each example contains a class index, article title, and article description.

### 4.2.2 Preparing the Dataset

The AG News data is stored in CSV format. We load the training and test files with pandas and assign descriptive column names before creating numerical text features.

In [28]:
from pathlib import Path
import urllib.request

AG_NEWS_DIR = Path("data/ag_news_csv")
AG_NEWS_DIR.mkdir(parents=True, exist_ok=True)

base_url = (
    "https://raw.githubusercontent.com/"
    "mhjabreel/CharCnn_Keras/master/data/ag_news_csv"
)

files = ["train.csv", "test.csv", "classes.txt"]

for filename in files:
    destination = AG_NEWS_DIR / filename

    if destination.exists():
        print(f"{filename} already exists.")
    else:
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(
            f"{base_url}/{filename}",
            destination
        )

print("AG News dataset ready.")

KeyboardInterrupt: 

In [29]:
from pathlib import Path

AG_NEWS_DIR = Path("data/ag_news_csv")

for file in AG_NEWS_DIR.glob("*"):
    print(file.name, f"{file.stat().st_size / (1024**2):.2f} MB")

train.csv 24.48 MB


In [30]:
import urllib.request

AG_NEWS_DIR.mkdir(parents=True, exist_ok=True)

urls = {
    "train.csv": (
        "https://github.com/tonyzhaozh/"
        "few-shot-learning/raw/main/data/agnews/train.csv"
    ),
    "test.csv": (
        "https://github.com/tonyzhaozh/"
        "few-shot-learning/raw/main/data/agnews/test.csv"
    ),
}

for filename, url in urls.items():
    destination = AG_NEWS_DIR / filename

    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, destination)

    size_mb = destination.stat().st_size / (1024 ** 2)
    print(f"{filename} complete: {size_mb:.2f} MB")

# The four class names used by AG News
classes = ["World", "Sports", "Business", "Sci/Tech"]

(AG_NEWS_DIR / "classes.txt").write_text(
    "\n".join(classes),
    encoding="utf-8"
)

print("AG News dataset ready.")

train.csv complete: 27.61 MB
test.csv complete: 1.74 MB
AG News dataset ready.


#### Load and Inspect the AG News Dataset

The AG News CSV files contain three columns:

1. class index
2. article title
3. article description

We load the training and test sets with pandas and inspect their structure before creating text features.

In [32]:
import pandas as pd

column_names = ["class", "title", "description"]

train_df = pd.read_csv(
    AG_NEWS_DIR / "train.csv",
    header=0,
    names=column_names
)

test_df = pd.read_csv(
    AG_NEWS_DIR / "test.csv",
    header=0,
    names=column_names
)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())

Training shape: (120000, 3)
Test shape: (7600, 3)


,class,title,description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


#### Prepare Text, Labels, and Tokens

The AG News class indices correspond to four categories:

1. World
2. Sports
3. Business
4. Sci/Tech

For classification, the article title and description are combined into a single text field.

The combined text is then tokenized into individual tokens. The training vocabulary is constructed from these tokens, keeping only terms that occur frequently enough in the training corpus.

In [33]:
import re

# Class-name mapping
labels = ["World", "Sports", "Business", "Sci/Tech"]

train_df["label"] = train_df["class"].map(
    lambda x: labels[int(x) - 1]
)

test_df["label"] = test_df["class"].map(
    lambda x: labels[int(x) - 1]
)

# Combine title and description
train_df["text"] = (
    train_df["title"].fillna("") + " " +
    train_df["description"].fillna("")
)

test_df["text"] = (
    test_df["title"].fillna("") + " " +
    test_df["description"].fillna("")
)

# Simple tokenizer
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

train_df["tokens"] = train_df["text"].map(tokenize)
test_df["tokens"] = test_df["text"].map(tokenize)

display(
    train_df[
        ["class", "label", "title", "text", "tokens"]
    ].head()
)

,class,label,title,text,tokens
0,3,Business,Wall St. Bears Claw Back Into the Black (Reuters),Wall St. Bears Claw Back Into the Black (Reute...,"[wall, st, bears, claw, back, into, the, black..."
1,3,Business,Carlyle Looks Toward Commercial Aerospace (Reu...,Carlyle Looks Toward Commercial Aerospace (Reu...,"[carlyle, looks, toward, commercial, aerospace..."
2,3,Business,Oil and Economy Cloud Stocks' Outlook (Reuters),Oil and Economy Cloud Stocks' Outlook (Reuters...,"[oil, and, economy, cloud, stocks, outlook, re..."
3,3,Business,Iraq Halts Oil Exports from Main Southern Pipe...,Iraq Halts Oil Exports from Main Southern Pipe...,"[iraq, halts, oil, exports, from, main, southe..."
4,3,Business,"Oil prices soar to all-time record, posing new...","Oil prices soar to all-time record, posing new...","[oil, prices, soar, to, all, time, record, pos..."


#### Build the Vocabulary

A vocabulary maps each retained token to a unique integer ID.

To reduce memory usage and remove very rare/noisy terms, only tokens occurring more than 10 times in the training corpus are retained.

A special `[UNK]` token represents words that are not present in the vocabulary.

In [34]:
# Build vocabulary from training tokens

threshold = 10

tokens = train_df["tokens"].explode().value_counts()

# Keep sufficiently frequent tokens
tokens = tokens[tokens > threshold]

# Add special unknown token
id_to_token = ["[UNK]"] + tokens.index.tolist()

# Map token -> integer ID
token_to_id = {
    word: i
    for i, word in enumerate(id_to_token)
}

vocabulary_size = len(id_to_token)

print(f"Vocabulary size: {vocabulary_size:,}")
print("First 10 vocabulary entries:")
print(id_to_token[:10])

Vocabulary size: 18,636
First 10 vocabulary entries:
['[UNK]', 'the', 'to', 'a', 'of', 'in', 'and', 's', 'on', 'for']


#### Create Bag-of-Words Feature Vectors

Each article is converted into a sparse Bag-of-Words representation.

The feature dictionary stores:

- the token ID as the key
- the token count in the article as the value

Tokens not found in the retained vocabulary are mapped to the `[UNK]` token.

In [35]:
from collections import defaultdict
from tqdm.auto import tqdm

tqdm.pandas()

def make_features(tokens, unk_id=0):
    vector = defaultdict(int)

    for token in tokens:
        token_id = token_to_id.get(token, unk_id)
        vector[token_id] += 1

    return vector


train_df["features"] = train_df["tokens"].progress_map(make_features)
test_df["features"] = test_df["tokens"].progress_map(make_features)

print("Example feature dictionary:")
print(dict(train_df["features"].iloc[0]))

  0%|          | 0/120000 [00:00<?, ?it/s]

  0%|          | 0/7600 [00:00<?, ?it/s]

Example feature dictionary:
{450: 2, 452: 1, 1648: 1, 14647: 1, 109: 1, 64: 1, 1: 1, 851: 1, 21: 2, 757: 1, 8212: 1, 388: 1, 7: 1, 10202: 1, 2897: 1, 4: 1, 5786: 1, 0: 1, 40: 1, 4063: 1, 794: 1, 339: 1}


#### Prepare Features and Class Labels for PyTorch

Multiclass classification in PyTorch uses zero-based class labels.

Therefore, the original AG News labels:

1, 2, 3, 4

are converted to:

0, 1, 2, 3.

The sparse feature dictionaries are converted to dense vectors only when needed, avoiding the large memory cost of storing the entire dense feature matrix.

In [36]:
# Convert a sparse feature dictionary into one dense vector

def make_dense(feats):
    x = np.zeros(vocabulary_size, dtype=np.float32)

    for token_id, count in feats.items():
        x[token_id] = count

    return x


# Convert labels from 1-4 to 0-3
y_train_mc = torch.tensor(
    train_df["class"].to_numpy() - 1,
    dtype=torch.long
)

y_test_mc = torch.tensor(
    test_df["class"].to_numpy() - 1,
    dtype=torch.long
)

# Inspect one example
x_example = make_dense(train_df["features"].iloc[0])

print("Feature vector shape:", x_example.shape)
print("Training labels shape:", y_train_mc.shape)
print("Test labels shape:", y_test_mc.shape)
print("Unique training labels:", torch.unique(y_train_mc))
print("First original label:", train_df["class"].iloc[0])
print("First PyTorch label:", y_train_mc[0].item())

Feature vector shape: (18636,)
Training labels shape: torch.Size([120000])
Test labels shape: torch.Size([7600])
Unique training labels: tensor([0, 1, 2, 3])
First original label: 3
First PyTorch label: 2


### 4.2.3 Multiclass Logistic Regression Using PyTorch

The multiclass logistic regression model consists of a single linear layer.

The input size is the vocabulary size and the output size is the number
of news categories (4).

Unlike binary classification, the model produces four scores for every
article. `CrossEntropyLoss` applies the multiclass loss and handles the
softmax operation internally.

The class with the highest output score is selected as the prediction.

In [37]:
from torch import nn, optim

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Book hyperparameters
lr = 1.0
n_epochs_mc = 5

n_examples_mc = len(train_df)
n_features_mc = vocabulary_size
n_classes = len(labels)

# Model
model_mc = nn.Linear(
    n_features_mc,
    n_classes
).to(device)

loss_func_mc = nn.CrossEntropyLoss()

optimizer_mc = optim.SGD(
    model_mc.parameters(),
    lr=lr
)

print(model_mc)
print("Device:", device)
print("Input features:", n_features_mc)
print("Output classes:", n_classes)

Linear(in_features=18636, out_features=4, bias=True)
Device: cpu
Input features: 18636
Output classes: 4


In [38]:
# Train multiclass logistic regression

indices_mc = np.arange(n_examples_mc)

model_mc.train()

for epoch in range(n_epochs_mc):

    np.random.shuffle(indices_mc)

    for i in tqdm(indices_mc, desc=f"Epoch {epoch + 1}"):

        # Convert only this article to a dense feature vector
        x_np = make_dense(train_df["features"].iloc[i])

        # Shape: [1, vocabulary_size]
        x = torch.from_numpy(x_np).unsqueeze(0).to(device)

        # Shape: [1]
        y_true = y_train_mc[i].unsqueeze(0).to(device)

        # 1. Clear previous gradients
        model_mc.zero_grad()

        # 2. Predict 4 class scores
        y_pred = model_mc(x)

        # 3. Compute multiclass cross-entropy loss
        loss = loss_func_mc(y_pred, y_true)

        # 4. Backpropagation
        loss.backward()

        # 5. Update parameters
        optimizer_mc.step()

Epoch 1:   0%|          | 0/120000 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/120000 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/120000 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/120000 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/120000 [00:00<?, ?it/s]

#### Evaluate the Multiclass Logistic Regression Model

The trained model is evaluated on the AG News test set.

For each article, the model produces four logits corresponding to the four news classes. The class with the highest logit is selected using `argmax`.

Softmax is not required for choosing the predicted class because applying softmax does not change which logit is the largest.

In [39]:
from sklearn.metrics import classification_report, accuracy_score

model_mc.eval()

y_pred_mc = []

with torch.no_grad():

    for feats in tqdm(test_df["features"], desc="Evaluating"):

        # Create one dense feature vector
        x_np = make_dense(feats)

        # Convert to PyTorch tensor
        x = torch.from_numpy(x_np).unsqueeze(0).to(device)

        # Get four class logits
        logits = model_mc(x)

        # Select class with highest score
        prediction = torch.argmax(logits, dim=1).item()

        y_pred_mc.append(prediction)


y_pred_mc = np.array(y_pred_mc)
y_true_mc = y_test_mc.numpy()

print(f"Accuracy: {accuracy_score(y_true_mc, y_pred_mc):.4f}")

print(
    classification_report(
        y_true_mc,
        y_pred_mc,
        target_names=labels,
        digits=4
    )
)

Evaluating:   0%|          | 0/7600 [00:00<?, ?it/s]

Accuracy: 0.8812
              precision    recall  f1-score   support

       World     0.8682    0.8879    0.8780      1900
      Sports     0.9593    0.9305    0.9447      1900
    Business     0.8512    0.8311    0.8410      1900
    Sci/Tech     0.8489    0.8753    0.8619      1900

    accuracy                         0.8812      7600
   macro avg     0.8819    0.8812    0.8814      7600
weighted avg     0.8819    0.8812    0.8814      7600



## 4.3 Summary

Chapter 4 implemented text classification algorithms introduced in
Chapters 2 and 3.

### Binary Classification — IMDb

Three models were implemented:

- Perceptron
- Logistic Regression from scratch using NumPy
- Logistic Regression using PyTorch

All achieved approximately 85–86% accuracy/F1.

### Multiclass Classification — AG News

A four-class logistic regression classifier was implemented using PyTorch.

Classes:

1. World
2. Sports
3. Business
4. Sci/Tech

Final test accuracy:

**88.12%**

The main PyTorch training workflow is:

**Forward Pass → Loss → Backpropagation → Optimizer Update**

PyTorch automatically calculates gradients through `loss.backward()`,
which eliminates the need to derive and implement gradient calculations
manually.

For binary classification, `BCEWithLogitsLoss` was used.
For multiclass classification, `CrossEntropyLoss` was used and `argmax`
selected the predicted class.

A major lesson from this chapter is that careful text preprocessing is
important and that PyTorch provides a reusable training structure that
can be extended to more complex neural networks.